# 머신러닝 기반 텍스트 분류
1. 데이터 준비 : 파일 로딩, 입/출력 데이터 추출, 학습/테스트 데이터로 분리, 특징 추출
2. 학습-평가
3. 배포 준비

### 1. 데이터 준비

In [1]:
import pandas as pd
datafile = './data/Korean_movie_reviews_2016.csv'
data_df = pd.read_csv(datafile)
data_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [2]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [3]:
# 입력/정답 데이터 추출
review_list = list(data_df.review) # 입력 데이터
label_list = list(data_df.label) # 출력 데이터
len(review_list), len(label_list)

(165384, 165384)

In [4]:
# 학습/테스트 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.1) # X는 벡터, y는 스칼라라서 대소문자 구분
len(train_X), len(test_X), len(train_y), len(test_y) # 테스트용으로 10% 사용

(148845, 16539, 148845, 16539)

In [5]:
# 한국어 토크나이저 정의
from konlpy.tag import Okt

def korean_tokenizer(text):
    my_tags = ['Noun', 'Adjective', 'Verb']
    my_stopwords = [] # 나중에 추가하기
    tokenizer = Okt()
    return [word for word, tag in tokenizer.pos(text) if tag in my_tags and word not in my_stopwords]

In [62]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000) # 고빈도 단어 1000개 추출(1000차원)
vectorizer.fit(train_X)

TfidfVectorizer(max_features=1000)

In [63]:
len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out()[:10]

(1000,
 array(['가고', '가는', '가면', '가볍', '가서', '가슴', '가장', '가족', '가지', '가치'],
       dtype=object))

In [64]:
# 학습 데이터 특징 추출
train_X_fv = vectorizer.transform(train_X)

In [65]:
# 테스트 데이터 특징 추출
test_X_fv = vectorizer.transform(test_X)

In [66]:
print(train_X_fv)

  (np.int32(0), np.int32(138))	0.8193072032384949
  (np.int32(0), np.int32(763))	0.5733547825923454
  (np.int32(1), np.int32(168))	0.5333003825461787
  (np.int32(1), np.int32(550))	0.51398470078893
  (np.int32(1), np.int32(710))	0.4578726915241174
  (np.int32(1), np.int32(763))	0.36601536691976433
  (np.int32(1), np.int32(926))	0.3283226749191473
  (np.int32(2), np.int32(269))	0.3530276789720722
  (np.int32(2), np.int32(294))	0.47064238761636507
  (np.int32(2), np.int32(637))	0.13062927340498376
  (np.int32(2), np.int32(757))	0.42109433110531636
  (np.int32(2), np.int32(829))	0.3779827161799578
  (np.int32(2), np.int32(958))	0.36511266070249887
  (np.int32(2), np.int32(980))	0.4281408288904184
  (np.int32(3), np.int32(21))	0.2369122279247295
  (np.int32(3), np.int32(76))	0.22558004696447365
  (np.int32(3), np.int32(100))	0.2219067538773106
  (np.int32(3), np.int32(140))	0.1781572050150462
  (np.int32(3), np.int32(156))	0.631245277022377
  (np.int32(3), np.int32(357))	0.2202432553495380

In [67]:
# 정답 데이터를 ndarray로 변환 np.array(sequence)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)
print(train_y[:10])

[0 0 0 1 1 0 0 0 0 0]


# 2. 머신러닝 - 모델 학습
1. 의사결정트리, Decision Tree (DT)
2. 랜덤포레스트, RandomForest (RF)

In [14]:
# 모델별 정확도를 dataframe으로 저장하여 비교
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])
score_df

,train,test


In [87]:
def get_scores(model, train_X, train_y, test_X, test_y):
    train_score = model.score(train_X, train_y) * 100
    test_score = model.score(test_X, test_y) * 100
    return train_score, test_score

### 1. Decision Tree

In [70]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [71]:
train_score, test_score = get_scores(dtc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

95.76606537001578 80.13785597678215


In [72]:
score_df.loc['DecisionTree'] = [train_score, test_score] # loc: 행 지정
score_df

,train,test
DecisionTree,95.766065,80.137856
RandomForest,95.765394,84.128424
NaiveBayes,85.247741,85.434428
LogisticRegression,86.182270,86.020920
SVM,86.141960,86.117661


### 2. RandomForest

In [82]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1)
rf.fit(train_X_fv, train_y)


RandomForestClassifier(n_jobs=-1)

In [83]:
train_score, test_score = get_scores(rf, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

score_df.loc['RandomForest'] = [train_score, test_score] # loc: 행 지정
score_df

95.76404985051565 83.91680270874902


,train,test
DecisionTree,95.766065,80.137856
RandomForest,95.764050,83.916803
NaiveBayes,85.247741,85.434428
LogisticRegression,86.182270,86.020920
SVM,86.141960,86.117661


### 3. 나이브 베이즈 분류

In [29]:
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [30]:
train_score, test_score = get_scores(mnb, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

score_df.loc['NaiveBayes'] = [train_score, test_score] # loc: 행 지정
score_df

85.24774093856024 85.43442771630691


,train,test
DecisionTree,98.089959,80.059254
RandomForest,98.089288,85.120019
NaiveBayes,85.247741,85.434428


### 4. 로지스틱 회귀 분석, logistic regression

In [33]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [34]:
train_score, test_score = get_scores(lr, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

score_df.loc['LogisticRegression'] = [train_score, test_score] # loc: 행 지정
score_df

86.18227014679701 86.02092024910817


,train,test
DecisionTree,98.089959,80.059254
RandomForest,98.089288,85.120019
NaiveBayes,85.247741,85.434428
LogisticRegression,86.182270,86.020920


### 5. 서포트벡터머신, SVM

In [37]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(train_X_fv, train_y)

LinearSVC()

In [38]:
train_score, test_score = get_scores(svm, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

score_df.loc['SVM'] = [train_score, test_score] # loc: 행 지정
score_df

86.14195975679398 86.11766128544652


,train,test
DecisionTree,98.089959,80.059254
RandomForest,98.089288,85.120019
NaiveBayes,85.247741,85.434428
LogisticRegression,86.182270,86.020920
SVM,86.141960,86.117661


In [39]:
score_df.sort_values(by='test', ascending=False)

,train,test
SVM,86.141960,86.117661
LogisticRegression,86.182270,86.020920
NaiveBayes,85.247741,85.434428
RandomForest,98.089288,85.120019
DecisionTree,98.089959,80.059254


### 6. 배포 준비
- 기능 구현
- 모델 저장

In [122]:
review = '이게 영화냐? 나도 만들겠다'

def analyze_sentiment(review):
    # 전처리 및 특징 벡터 추출
    review_fv = vectorizer.transform([review])
    # print(review_fv)

    result = rf.predict(review_fv)
    # print(result)

    show = '긍정' if result[0] >= 0.5 else '부정'
    return show

show = analyze_sentiment(review)
print(f'{review} -> {show}')

이게 영화냐? 나도 만들겠다 -> 긍정


In [117]:
reviews = [
    '영화가 너무 재미있다.',
    '개꿀잼',
    '이게 영화냐? 나도 만들겠다',
    '영화가 재미없다'
]

for review in reviews:
    print(f'{review} -> {analyze_sentiment(review)}')

영화가 너무 재미있다. -> 긍정
개꿀잼 -> 긍정
이게 영화냐? 나도 만들겠다 -> 긍정
영화가 재미없다 -> 긍정


In [121]:
import joblib

vectorizer_file = './model/sa_movie_vectorizer.pkl'
joblib.dump(vectorizer, vectorizer_file)

model_file = './model/sa_movie_model.pkl'
joblib.dump(rf, model_file)

['./model/sa_movie_model.pkl']